# Cold Start  — Pacientes sin historial


## Objetivo

Construir un componente predictivo para pacientes que **no tienen historial previo**.

### Entrada real del paciente nuevo

El sistema solo necesita:

```text
ID, Edad, Comunas, Sexo
```

- `ID`: solo trazabilidad. **No entra al modelo.**
- `Edad`
- `Sexo`
- `Comuna`

### Salidas

El Cold Start entrega dos estimaciones iniciales:

1. **Cantidad mensual estimada de cilindros**
2. **Intervalo estimado hasta la próxima necesidad**, expresado en:
   - días aproximados;
   - rango operacional.

El entrenamiento se realiza una única vez con pacientes históricos, pero las variables predictoras siguen siendo exclusivamente:

```text
Edad + Sexo + Comuna
```

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# PARTE 1 — Entrenar el Cold Start

Para entrenar necesitamos un dataset histórico que ya contenga el comportamiento real observado de cada paciente.

El archivo esperado es `dataset_completo_cruzado.csv`.

Del dataset histórico solo usaremos como **features**:

```text
Edad
Sexo
Comuna
```

Las columnas históricas se usan únicamente para crear los targets.

In [ ]:
uploaded = files.upload()

csv_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".csv")
]

if len(csv_files) != 1:
    raise ValueError("Sube exactamente un CSV histórico.")

train_file = csv_files[0]

hist = pd.read_csv(
    train_file,
    sep=None,
    engine="python"
)

print(f"Archivo histórico: {train_file}")
print(f"Filas: {len(hist):,}")
display(hist.head())

Saving dataset_completo_cruzado.csv to dataset_completo_cruzado.csv
Archivo histórico: dataset_completo_cruzado.csv
Filas: 1,048


,Edad,Comunas,Sexo,ID,ene,feb,mar,abr,may,jun,...,ago,sep,oct,nov,Suma_total,id_num,n_despachos,meses_activos,intervalo_median,cv_intervalo
0,85.0,SAN JAVIER,Femenino,1,0,0,0,0,0,0,...,0,0,1,0,1,1,3.0,3.0,127.5,0.249567
1,84.0,PUCON,Masculino,2,2,2,3,5,4,4,...,4,4,5,4,42,2,42.0,11.0,7.0,0.360022
2,101.0,NUNOA,Femenino,4,0,0,0,0,0,0,...,0,1,0,1,2,4,3.0,3.0,155.0,1.003635
3,92.0,VITACURA,Femenino,6,2,1,1,1,1,1,...,0,2,0,1,10,6,10.0,8.0,31.0,0.633324
4,103.0,PROVIDENCIA,Femenino,8,0,2,0,1,0,2,...,2,2,2,1,12,8,15.0,9.0,19.0,0.531362


## 1. Normalización del dataset histórico

In [ ]:
hist.columns = [str(c).strip() for c in hist.columns]

hist = hist.rename(columns={
    "Edad": "edad",
    "Comunas": "comuna",
    "Comuna": "comuna",
    "Sexo": "sexo",
    "ID": "id",
    "Suma_total": "suma_total"
})

required_training = [
    "edad",
    "comuna",
    "sexo",
    "suma_total",
    "meses_activos",
    "intervalo_median"
]

missing = [
    c for c in required_training
    if c not in hist.columns
]

if missing:
    raise ValueError(
        f"El dataset histórico no contiene las columnas necesarias: {missing}"
    )

for c in [
    "edad",
    "suma_total",
    "meses_activos",
    "intervalo_median"
]:
    hist[c] = pd.to_numeric(
        hist[c],
        errors="coerce"
    )

hist["comuna"] = (
    hist["comuna"]
    .astype("string")
    .str.strip()
    .str.upper()
)

hist["sexo"] = (
    hist["sexo"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Dataset histórico normalizado.")

Dataset histórico normalizado.


## 2. Crear targets de entrenamiento

### Cantidad

Usamos como objetivo la cantidad promedio de cilindros por mes activo:

```text
cantidad_mensual = suma_total / meses_activos
```

### Tiempo

Usamos el intervalo mediano histórico entre entregas:

```text
intervalo_dias = intervalo_median
```

Estas variables existen solo durante el entrenamiento.  
**No se pedirán a un paciente nuevo.**

In [ ]:
hist["target_cantidad"] = np.where(
    hist["meses_activos"] > 0,
    hist["suma_total"] / hist["meses_activos"],
    np.nan
)

hist["target_dias"] = np.where(
    hist["intervalo_median"] > 0,
    hist["intervalo_median"],
    np.nan
)

print("Target cantidad:")
display(hist["target_cantidad"].describe())

print("\nTarget días:")
display(hist["target_dias"].describe())

Target cantidad:


,target_cantidad
count,1048.000000
mean,1.801912
std,1.210427
min,0.090909
25%,1.000000
50%,1.454545
75%,2.363636
max,8.727273



Target días:


,target_dias
count,1048.000000
mean,22.947996
std,20.137100
min,3.000000
25%,10.000000
50%,17.000000
75%,29.000000
max,159.000000


## 3. Definir las únicas features del Cold Start

In [ ]:
NUMERIC_FEATURES = ["edad"]
CATEGORICAL_FEATURES = ["sexo", "comuna"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def make_preprocessor():
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    try:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder)
    ])

    return ColumnTransformer([
        ("num", numeric, NUMERIC_FEATURES),
        ("cat", categorical, CATEGORICAL_FEATURES)
    ])

def make_model():
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", RandomForestRegressor(
            n_estimators=400,
            max_depth=12,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

print("Features reales del modelo:", FEATURES)

Features reales del modelo: ['edad', 'sexo', 'comuna']


## 4. Entrenar modelo de cantidad

In [ ]:
df_q = hist.dropna(
    subset=FEATURES + ["target_cantidad"]
).copy()

X_q = df_q[FEATURES]
y_q = df_q["target_cantidad"]

Xq_train, Xq_test, yq_train, yq_test = train_test_split(
    X_q,
    y_q,
    test_size=0.20,
    random_state=RANDOM_STATE
)

modelo_cantidad = make_model()

modelo_cantidad.fit(
    Xq_train,
    yq_train
)

pred_q = np.maximum(
    0,
    modelo_cantidad.predict(Xq_test)
)

mae_q = mean_absolute_error(
    yq_test,
    pred_q
)

rmse_q = np.sqrt(
    mean_squared_error(
        yq_test,
        pred_q
    )
)

r2_q = r2_score(
    yq_test,
    pred_q
)

print("MODELO CANTIDAD")
print(f"MAE : {mae_q:.4f}")
print(f"RMSE: {rmse_q:.4f}")
print(f"R²  : {r2_q:.4f}")

MODELO CANTIDAD
MAE : 0.9376
RMSE: 1.2308
R²  : 0.0119


## 5. Entrenar modelo de tiempo

In [ ]:
df_t = hist.dropna(
    subset=FEATURES + ["target_dias"]
).copy()

X_t = df_t[FEATURES]
y_t = df_t["target_dias"]

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_t,
    y_t,
    test_size=0.20,
    random_state=RANDOM_STATE
)

modelo_dias = make_model()

modelo_dias.fit(
    Xt_train,
    yt_train
)

pred_t = np.maximum(
    1,
    modelo_dias.predict(Xt_test)
)

mae_t = mean_absolute_error(
    yt_test,
    pred_t
)

rmse_t = np.sqrt(
    mean_squared_error(
        yt_test,
        pred_t
    )
)

r2_t = r2_score(
    yt_test,
    pred_t
)

print("MODELO TIEMPO")
print(f"MAE : {mae_t:.4f} días")
print(f"RMSE: {rmse_t:.4f} días")
print(f"R²  : {r2_t:.4f}")

MODELO TIEMPO
MAE : 13.3207 días
RMSE: 20.8064 días
R²  : -0.0571


## 6. Reentrenar con todos los datos históricos

Una vez evaluados, los dos modelos se entrenan nuevamente usando todos los pacientes históricos disponibles.

In [ ]:
modelo_cantidad.fit(
    df_q[FEATURES],
    df_q["target_cantidad"]
)

modelo_dias.fit(
    df_t[FEATURES],
    df_t["target_dias"]
)

print("Modelos finales entrenados.")

Modelos finales entrenados.


## 7. Guardar modelo Cold Start

In [ ]:
bundle = {
    "modelo_cantidad": modelo_cantidad,
    "modelo_dias": modelo_dias,
    "features": FEATURES,
    "metricas": {
        "cantidad": {
            "MAE": float(mae_q),
            "RMSE": float(rmse_q),
            "R2": float(r2_q)
        },
        "tiempo": {
            "MAE_dias": float(mae_t),
            "RMSE_dias": float(rmse_t),
            "R2": float(r2_t)
        }
    }
}

joblib.dump(
    bundle,
    "modelo_cold_start.joblib"
)

print("Guardado: modelo_cold_start.joblib")

Guardado: modelo_cold_start.joblib


# PARTE 2 — Usar Cold Start con pacientes nuevos

Usaremos un archivo CSV que sólo contenga:

```text
ID, Edad, Comunas, Sexo
```


In [ ]:
uploaded_new = files.upload()

csv_files_new = [
    name for name in uploaded_new.keys()
    if name.lower().endswith(".csv")
]

if len(csv_files_new) != 1:
    raise ValueError(
        "Sube exactamente un CSV de pacientes nuevos."
    )

new_file = csv_files_new[0]

nuevos = pd.read_csv(
    new_file,
    sep=None,
    engine="python"
)

print(f"Archivo nuevos: {new_file}")
display(nuevos.head())

Saving pacientes_sinteticos_cold_start.csv to pacientes_sinteticos_cold_start (1).csv
Archivo nuevos: pacientes_sinteticos_cold_start (1).csv


,﻿ID,Edad,Comunas,Sexo
0,CS001,0,VALLENAR,Masculino
1,CS002,1,BUIN,Femenino
2,CS003,2,VALDIVIA,Masculino
3,CS004,8,PUENTE ALTO,Femenino
4,CS005,15,SANTIAGO,Masculino


## 8. Normalizar pacientes nuevos

In [ ]:
nuevos.columns = [
    str(c).strip()
    for c in nuevos.columns
]

nuevos = nuevos.rename(columns={
    "﻿ID": "id",
    "Edad": "edad",
    "Comunas": "comuna",
    "Comuna": "comuna",
    "Sexo": "sexo"
})

required_new = [
    "id",
    "edad",
    "comuna",
    "sexo"
]

missing = [
    c for c in required_new
    if c not in nuevos.columns
]

if missing:
    raise ValueError(
        f"Faltan columnas: {missing}"
    )

nuevos["edad"] = pd.to_numeric(
    nuevos["edad"],
    errors="coerce"
)

nuevos["comuna"] = (
    nuevos["comuna"]
    .astype("string")
    .str.strip()
    .str.upper()
)

nuevos["sexo"] = (
    nuevos["sexo"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Pacientes nuevos listos.")
display(nuevos.head())

Pacientes nuevos listos.


,id,edad,comuna,sexo
0,CS001,0,VALLENAR,MASCULINO
1,CS002,1,BUIN,FEMENINO
2,CS003,2,VALDIVIA,MASCULINO
3,CS004,8,PUENTE ALTO,FEMENINO
4,CS005,15,SANTIAGO,MASCULINO


## 9. Predicción Cold Start

In [ ]:
X_new = nuevos[
    FEATURES
].copy()

cantidad_raw = np.maximum(
    0,
    modelo_cantidad.predict(X_new)
)

dias_raw = np.maximum(
    1,
    modelo_dias.predict(X_new)
)

cantidad_estimada = np.maximum(
    0,
    np.rint(cantidad_raw)
).astype(int)

dias_estimados = np.maximum(
    1,
    np.rint(dias_raw)
).astype(int)

def rango_desde_dias(d):
    if d <= 5:
        return "Muy pronto"
    elif d <= 8:
        return "Semanal"
    elif d <= 15:
        return "Quincenal"
    elif d <= 30:
        return "Mensual"
    return "Largo plazo"

rango_estimado = [
    rango_desde_dias(d)
    for d in dias_estimados
]

resultado = nuevos[
    [
        "id",
        "edad",
        "comuna",
        "sexo"
    ]
].copy()

resultado["cantidad_mensual_estimada"] = cantidad_estimada
resultado["dias_estimados"] = dias_estimados
resultado["rango_estimado"] = rango_estimado
resultado["estrategia"] = "COLD_START"

display(resultado)

,id,edad,comuna,sexo,cantidad_mensual_estimada,dias_estimados,rango_estimado,estrategia
0,CS001,0,VALLENAR,MASCULINO,1,26,Mensual,COLD_START
1,CS002,1,BUIN,FEMENINO,1,27,Mensual,COLD_START
2,CS003,2,VALDIVIA,MASCULINO,2,26,Mensual,COLD_START
3,CS004,8,PUENTE ALTO,FEMENINO,1,27,Mensual,COLD_START
4,CS005,15,SANTIAGO,MASCULINO,2,26,Mensual,COLD_START
5,CS006,24,ANTOFAGASTA,FEMENINO,1,28,Mensual,COLD_START
6,CS007,31,CALAMA,MASCULINO,2,20,Mensual,COLD_START
7,CS008,38,LA SERENA,FEMENINO,1,23,Mensual,COLD_START
8,CS009,42,RECOLETA,MASCULINO,1,20,Mensual,COLD_START
9,CS010,47,NUNOA,FEMENINO,1,21,Mensual,COLD_START


## 10. Función individual

In [ ]:
def predecir_cold_start(
    edad,
    sexo,
    comuna
):
    entrada = pd.DataFrame([{
        "edad": edad,
        "sexo": str(sexo).strip().upper(),
        "comuna": str(comuna).strip().upper()
    }])

    cantidad = float(
        modelo_cantidad.predict(
            entrada
        )[0]
    )

    dias = float(
        modelo_dias.predict(
            entrada
        )[0]
    )

    cantidad = int(
        max(
            0,
            round(cantidad)
        )
    )

    dias = int(
        max(
            1,
            round(dias)
        )
    )

    return {
        "cantidad_mensual_estimada": cantidad,
        "dias_estimados": dias,
        "rango_estimado": rango_desde_dias(dias),
        "estrategia": "COLD_START"
    }

## 11. Exportar predicciones

In [ ]:
resultado.to_csv(
    "predicciones_cold_start.csv",
    index=False
)

print(
    "Guardado: predicciones_cold_start.csv"
)

Guardado: predicciones_cold_start.csv


## 12. Descargar artefactos

In [ ]:
files.download(
    "modelo_cold_start.joblib"
)

# Opcional:
# files.download("predicciones_cold_start.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Flujo final

```text
PACIENTE NUEVO
ID + Edad + Sexo + Comuna
          |
          | ID solo trazabilidad
          v
Edad + Sexo + Comuna
          |
          v
COLD START
          |
          +--> cantidad mensual estimada
          |
          +--> días estimados
          |
          +--> rango estimado
```

No existe ninguna dependencia de historial para el paciente nuevo.